# Analyse exploratoire — dataset TMDB (9 837 films)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/movies.csv", engine="python", on_bad_lines="skip")
print(f"{len(df)} films chargés")

## 1. Vue d'ensemble

In [ ]:
df.head(3)

In [ ]:
df.dtypes

## 2. Données manquantes

In [ ]:
missing = pd.DataFrame(
    {
        "manquants": df.isna().sum(),
        "% manquants": (df.isna().mean() * 100).round(1),
    }
).sort_values("manquants", ascending=False)

display(missing)

# Vides (chaînes vides) pour les colonnes texte
text_cols = df.select_dtypes(include="object").columns
empty_strings = {col: (df[col].str.strip() == "").sum() for col in text_cols}
print("\nChaînes vides par colonne texte :")
for col, n in empty_strings.items():
    if n:
        print(f"  {col}: {n}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
missing["% manquants"].plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("% de valeurs manquantes")
ax.set_title("Données manquantes par colonne")
plt.tight_layout()
plt.show()

## 3. Distribution des notes (`Vote_Average`)

In [ ]:
df["Vote_Average"] = pd.to_numeric(df["Vote_Average"], errors="coerce")
print(df["Vote_Average"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3))

sns.histplot(df["Vote_Average"].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution des notes")
axes[0].set_xlabel("Note moyenne")

sns.boxplot(x=df["Vote_Average"].dropna(), ax=axes[1])
axes[1].set_title("Boîte à moustaches — notes")
axes[1].set_xlabel("Note moyenne")

plt.tight_layout()
plt.show()

## 4. Distribution de la popularité (`Popularity`)

In [ ]:
df["Popularity"] = pd.to_numeric(df["Popularity"], errors="coerce")
print(df["Popularity"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3))

sns.histplot(df["Popularity"].dropna(), bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribution de la popularité (brute)")

log_pop = np.log1p(df["Popularity"].dropna())
sns.histplot(log_pop, bins=40, kde=True, ax=axes[1])
axes[1].set_title("Distribution de la popularité (log)")

plt.tight_layout()
plt.show()

## 5. Nombre de votes (`Vote_Count`)

In [ ]:
df["Vote_Count"] = pd.to_numeric(df["Vote_Count"], errors="coerce")
print(df["Vote_Count"].describe().round(0))

fig, ax = plt.subplots(figsize=(7, 3))
sns.histplot(df["Vote_Count"].dropna(), bins=60, log_scale=(False, True), ax=ax)
ax.set_title("Distribution du nombre de votes (axe Y log)")
ax.set_xlabel("Nombre de votes")
plt.tight_layout()
plt.show()

## 6. Top genres

In [ ]:
genre_counts = df["Genre"].dropna().str.split(",").explode().str.strip().value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
genre_counts.head(20).plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Top 20 genres")
ax.set_xlabel("Nombre de films")
plt.tight_layout()
plt.show()

## 7. Langues originales

In [ ]:
lang_counts = df["Original_Language"].value_counts()
print(f"{lang_counts.shape[0]} langues distinctes\n")

fig, ax = plt.subplots(figsize=(8, 4))
lang_counts.head(15).plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Top 15 langues originales")
ax.set_xlabel("Nombre de films")
plt.tight_layout()
plt.show()

## 8. Films par année de sortie

In [ ]:
df["year"] = pd.to_datetime(df["Release_Date"], errors="coerce").dt.year
year_counts = df["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 3))
year_counts.plot.bar(ax=ax, width=0.9)
ax.set_title("Nombre de films par année")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre de films")
ax.tick_params(axis="x", labelrotation=45)
plt.tight_layout()
plt.show()

## 10. Films sans synopsis (non indexés pour les embeddings)

In [ ]:
no_overview = df[df["Overview"].isna() | (df["Overview"].str.strip() == "")]
print(f"Films sans synopsis : {len(no_overview)} / {len(df)} ({len(no_overview)/len(df)*100:.1f}%)")
no_overview[["Title", "Release_Date", "Original_Language", "Genre"]].head(10)

## 11. Détection d'anomalies

### 11.1 Valeurs déplacées — mauvaise colonne

In [ ]:
# URL dans Original_Language
lang_url = df[df["Original_Language"].str.startswith("http", na=False)]
print(f"URLs dans Original_Language : {len(lang_url)}")
display(lang_url[["Title", "Release_Date", "Original_Language", "Genre", "Poster_Url"]])

# URL dans Overview
ov_url = df[df["Overview"].str.startswith("http", na=False)]
print(f"\nURLs dans Overview : {len(ov_url)}")
display(ov_url[["Title", "Overview"]].head())

# Valeur numérique brute dans Overview (ex: "35")
ov_numeric = df[df["Overview"].str.match(r"^\d+\.?\d*$", na=False)]
print(f"\nOverview purement numérique : {len(ov_numeric)}")
display(ov_numeric[["Title", "Release_Date", "Overview"]].head())

### 11.2 Doublons de titre

In [ ]:
dup = df[df.duplicated("Title", keep=False)].sort_values("Title")
print(f"{df['Title'].duplicated().sum()} titres dupliqués, {len(dup)} lignes concernées")

# Les remakes légitimes ont des dates différentes — on cherche les vrais doublons
same_date_dup = dup[dup.duplicated(["Title", "Release_Date"], keep=False)]
print(f"\nDoublons titre + date identiques (copie exacte probable) : {len(same_date_dup)}")
display(same_date_dup[["Title", "Release_Date", "Original_Language", "Vote_Count"]].head(10))

# Afficher quelques remakes légitimes (même titre, dates différentes)
print("\nExemple de remakes légitimes (même titre, dates distinctes) :")
display(
    dup[~dup.duplicated(["Title", "Release_Date"], keep=False)]
    .groupby("Title")
    .apply(lambda g: g[["Release_Date", "Original_Language"]])
    .head(12)
)

### 11.3 Langues suspectes (valeurs inhabituellement longues)

In [ ]:
lang_series = df["Original_Language"]

# Distribution de la longueur des codes langue (attendu : 2 chars ISO 639-1)
lang_len = lang_series.str.len()
print("Distribution des longueurs de codes langue :")
print(lang_len.value_counts().sort_index())

# Codes hors norme ISO (> 3 chars)
weird_langs = df[lang_series.str.len() > 3]
print(f"\nCodes langue > 3 caractères : {len(weird_langs)}")
display(weird_langs[["Title", "Original_Language", "Poster_Url"]].head(10))

fig, ax = plt.subplots(figsize=(6, 3))
lang_len.value_counts().sort_index().plot.bar(ax=ax)
ax.set_title("Longueur des codes langue")
ax.set_xlabel("Nombre de caractères")
ax.set_ylabel("Nombre de films")
plt.tight_layout()
plt.show()

### 11.4 Films sans votes (Vote_Count = 0)

In [ ]:
vc = pd.to_numeric(df["Vote_Count"], errors="coerce")
no_votes = df[vc == 0]
print(f"Films avec Vote_Count = 0 : {len(no_votes)}")

# Ces films ont-ils quand même une note ?
has_score = no_votes[pd.to_numeric(no_votes["Vote_Average"], errors="coerce") > 0]
print(f"  dont {len(has_score)} ont une Vote_Average > 0 malgré 0 votes (incohérence)")
display(has_score[["Title", "Release_Date", "Vote_Average", "Vote_Count"]].head(10))

### 11.5 Synopsis générique / placeholder

In [ ]:
PLACEHOLDERS = ["plot unknown", "no overview", "n/a", "tbd", "to be announced"]

mask = df["Overview"].str.lower().str.strip().isin(PLACEHOLDERS)
placeholder_ov = df[mask]
print(f"Synopsis placeholder : {len(placeholder_ov)}")
display(placeholder_ov[["Title", "Overview"]].head())

# Synopsis très courts (< 30 chars) mais non-nuls
short_ov = df[df["Overview"].str.len().between(1, 29)]
print(f"\nSynopsis très courts (1–29 chars) : {len(short_ov)}")
display(short_ov[["Title", "Overview"]].head(10))

### 11.6 Récapitulatif des anomalies

In [ ]:
vc = pd.to_numeric(df["Vote_Count"], errors="coerce")
lang_s = df["Original_Language"].dropna()

anomalies = {
    "URL dans Original_Language": df["Original_Language"].str.startswith("http", na=False).sum(),
    "Overview purement numérique": df["Overview"].str.match(r"^\d+\.?\d*$", na=False).sum(),
    "Synopsis placeholder": df["Overview"].str.lower().str.strip().isin(PLACEHOLDERS).sum(),
    "Synopsis très court (1–29 chars)": df["Overview"].str.len().between(1, 29).sum(),
    "Code langue > 3 chars": (lang_s.str.len() > 3).sum(),
    "Vote_Count = 0": (vc == 0).sum(),
    "Vote_Count=0 mais Vote_Average > 0": (
        (vc == 0) & (pd.to_numeric(df["Vote_Average"], errors="coerce") > 0)
    ).sum(),
    "Doublons titre+date": df.duplicated(["Title", "Release_Date"]).sum(),
    "Titre manquant": df["Title"].isna().sum(),
    "Poster_Url manquante": df["Poster_Url"].isna().sum(),
}

summary = pd.DataFrame.from_dict(anomalies, orient="index", columns=["count"])
summary["% du dataset"] = (summary["count"] / len(df) * 100).round(2)
display(summary.sort_values("count", ascending=False))